# Агент для GitHub-репозитория на локальной Qwen3.8-27B (Kaggle T4 x2)

Адаптация вашего ноутбука `benchmark-task-agentic-github` под локальную модель. **Перед запуском:** GPU T4 x2, Internet On,
подключите Dataset (или вывод ноутбука) с пакетом `llama-server` + модель, как в основном чат-ноутбуке.

**Токен GitHub** (Add-ons → Secrets → `GITHUB_TOKEN`, fine-grained, только на нужный репозиторий):
`Contents: Read`, `Metadata: Read`, `Issues: Read and write` (комментарии и новые issues), `Codespaces: Read and write` (только если нужны проверки в Codespaces).
Для публичного репозитория без записи токен не обязателен.

**Что делает агент**
1. Клонирует репозиторий в `/tmp` и читает все issues через GitHub API (сначала `priority:p0`, `p1`, `p2`, потом остальные).
2. Для каждого issue создаёт локальную ветку `agent/issue-N`; модель решает его инструментами: читает код, правит, запускает тесты, коммитит.
3. Работа идёт раундами (до `MAX_ROUNDS`); каждый раунд — новый чат с фактами из git, поэтому контекст не разрастается.
4. Решение засчитывается, только если есть реальный diff и принят финальный отчёт. Если правка не нужна, агент обязан обосновать это, а модель-ревьюер (та же) оценит обоснование.
5. **Комментарий в issue:** при итогах `solved`/`partial` система публикует отчёт агента с патчем (повторный запуск обновляет тот же комментарий, а не плодит новые). Модель может сама задать вопрос людям через `comment_issue` (лимит на issue).
6. **Новые issues:** если модель находит отдельную проблему, она заводит issue (`priority:p0/p1/p2` + метка `qwen-agent`), дубликаты (в том числе закрытых issues) отклоняются, есть лимит за прогон. Новые issues в этом же прогоне не решаются.
7. **Codespaces:** если локально проверить нельзя, модель вызывает `run_in_codespace`. Среда создаётся лениво, правки переносятся в неё автоматически (без push), в конце прогона Codespace останавливается. Тратит вашу квоту Codespaces.
8. В конце — отчёт `agent_report.md` (что сделано, что не удалось, почему, какие комментарии и issues созданы) и патчи `patches/issue-N.patch`.

**Что агент по-прежнему НЕ делает:** не выполняет `git push`, не создаёт Pull Request, не меняет код в вашем репозитории. Записи в GitHub — только комментарии, новые issues и временный Codespace.
Режим без записи: `GITHUB_WRITE = False` (комментарии и новые issues сохраняются черновиками в `agent_out/drafts/`).

**Из вашего ноутбука перенесено:** текстовый протокол `@@CALL/@@ARG/@@END`, бюджет шагов с подсказками, раунды в новых чатах с «проверенными системой» фактами, правило «нет коммитов — нет зачёта», путь «правка не нужна» с проверкой, приоритеты issues, жизненный цикл Codespaces и повторные попытки ssh.

## 1. Модель и сервер

In [ ]:
# ===== МОДЕЛЬ И СЕРВЕР =====
REPO_ID   = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF"   # используется, только если пакета в /kaggle/input нет
QUANT     = "UD-Q4_K_XL"
USE_VISION = False            # для работы с кодом картинки не нужны; без проектора остаётся больше памяти под контекст
MODEL_DIR = "/tmp/models"
LLAMA_DIR = "/tmp/llama.cpp"
WORK_DIR  = "/kaggle/working"
PORT      = 8080
# (n_ctx, тип KV-кэша): сервер пробует сверху вниз, пока модель не поместится
CTX_CANDIDATES = [(262144, "q8_0"), (262144, "q4_0"), (131072, "q8_0"), (65536, "q8_0")]
TENSOR_SPLIT   = "47,53"

In [ ]:
%%bash
set -e
mkdir -p /tmp/llama-bin
PACKED=$(find /kaggle/input -name llama-server -type f 2>/dev/null | head -n 1)
if [ -n "$PACKED" ]; then
  echo "Нашёл готовый llama-server в Dataset: $PACKED"
  cp "$(dirname "$PACKED")"/llama-server "$(dirname "$PACKED")"/*.so* /tmp/llama-bin/ 2>/dev/null || cp "$PACKED" /tmp/llama-bin/
else
  nvidia-smi --query-gpu=name,memory.total --format=csv
  which nvcc || echo "nvcc не найден!"
  rm -rf /tmp/llama.cpp
  git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
  cd /tmp/llama.cpp
  # GGML_CUDA_NO_VMM: в образе Kaggle нет заглушки libcuda, без этого флага cmake падает на CUDA::cuda_driver
  # GGML_NATIVE=OFF: бинарник не привязан к конкретному процессору (важно, если его потом сохраняем в Dataset)
  cmake -B build -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES=75 \
        -DGGML_NATIVE=OFF -DGGML_AVX2=ON -DGGML_FMA=ON -DGGML_F16C=ON \
        -DLLAMA_BUILD_TESTS=OFF -DCMAKE_BUILD_TYPE=Release
  cmake --build build -j"$(nproc)" --target llama-server
  cp build/bin/llama-server build/bin/*.so* /tmp/llama-bin/
fi
chmod +x /tmp/llama-bin/llama-server
ls -la /tmp/llama-bin
LD_LIBRARY_PATH=/tmp/llama-bin /tmp/llama-bin/llama-server --version 2>&1 | head -n 3

In [ ]:
import os, glob
from huggingface_hub import HfApi, hf_hub_download

os.makedirs(MODEL_DIR, exist_ok=True)

def pick_mmproj(names):
    # на T4 нет аппаратной поддержки bf16, поэтому F16 предпочтительнее
    return (next((f for f in names if "f16" in f.lower() and "bf16" not in f.lower()), None)
            or next((f for f in names if "bf16" in f.lower()), None)
            or (names[0] if names else None))

def download_from_hf():
    files = HfApi().list_repo_files(REPO_ID)
    ggufs = sorted(f for f in files if f.endswith(".gguf") and "mmproj" not in f.lower() and QUANT in f)
    if not ggufs:
        raise RuntimeError(f"Не нашёл файлов с '{QUANT}' в {REPO_ID}. Доступно: " +
                           ", ".join(sorted(f for f in files if f.endswith('.gguf'))[:30]))
    mm_pick = pick_mmproj([f for f in files if "mmproj" in f.lower() and f.endswith(".gguf")])
    print("Скачиваю:", ggufs, mm_pick)
    paths = [hf_hub_download(REPO_ID, f, local_dir=MODEL_DIR) for f in ggufs]
    mm_path = hf_hub_download(REPO_ID, mm_pick, local_dir=MODEL_DIR) if mm_pick else None
    return paths[0], mm_path   # для шардов llama.cpp сам найдёт остальные части

# 1) если модель уже лежит в подключённом Dataset, скачивать ничего не нужно
local = sorted(glob.glob("/kaggle/input/**/*.gguf", recursive=True))
local_mm = [p for p in local if "mmproj" in os.path.basename(p).lower()]
local_models = [p for p in local if p not in local_mm]
local_models.sort(key=lambda p: QUANT not in os.path.basename(p))   # файл с нужной квантизацией первым

if local_models:
    MODEL_PATH, MMPROJ_PATH = local_models[0], pick_mmproj(local_mm)
    print("Беру модель из Dataset:", MODEL_PATH)
else:
    MODEL_PATH, MMPROJ_PATH = download_from_hf()
print("Модель:", MODEL_PATH)
print("mmproj:", MMPROJ_PATH)

In [ ]:
if not USE_VISION:
    MMPROJ_PATH = None
    print("Проектор для изображений отключён (USE_VISION=False)")

In [ ]:
import os, subprocess, time, requests, re

SERVER_BIN = "/tmp/llama-bin/llama-server"
API        = f"http://127.0.0.1:{PORT}"
LOG_FILE   = "/tmp/llama-server.log"
server_proc = None

def stop_server():
    global server_proc
    if server_proc is not None and server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            server_proc.kill()
    server_proc = None
    time.sleep(3)   # дать драйверу освободить видеопамять

def start_server(n_ctx, kv, timeout=900):
    global server_proc
    stop_server()
    cmd = [SERVER_BIN, "-m", MODEL_PATH, "-c", str(n_ctx), "-ngl", "99", "-fa", "on",
           "-ctk", kv, "-ctv", kv, "-ts", TENSOR_SPLIT, "-np", "1",
           "--jinja", "--reasoning-format", "deepseek",
           "--host", "127.0.0.1", "--port", str(PORT)]
    if MMPROJ_PATH:
        cmd += ["--mmproj", MMPROJ_PATH]
    env = {**os.environ, "LD_LIBRARY_PATH": "/tmp/llama-bin:" + os.environ.get("LD_LIBRARY_PATH", "")}
    server_proc = subprocess.Popen(cmd, stdout=open(LOG_FILE, "w"), stderr=subprocess.STDOUT, env=env)
    t0 = time.time()
    while time.time() - t0 < timeout:
        if server_proc.poll() is not None:
            return False
        try:
            if requests.get(API + "/health", timeout=3).status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(3)
    return False

def log_tail(n=12):
    lines = open(LOG_FILE, errors="ignore").read().splitlines()
    bad = [l for l in lines if re.search(r"error|out of memory|failed|unknown model architecture", l, re.I)]
    print("\n".join((bad or lines)[-n:]))

N_CTX, KV_TYPE = None, None
for n_ctx, kv in CTX_CANDIDATES:
    print(f"▶ пробую контекст {n_ctx} токенов, KV-кэш {kv} …")
    if start_server(n_ctx, kv):
        N_CTX, KV_TYPE = n_ctx, kv
        break
    print("  не вышло:"); log_tail()
if N_CTX is None:
    raise RuntimeError("Сервер не запустился ни с одной конфигурацией. Смотрите /tmp/llama-server.log "
                       "(частые причины: старая сборка llama.cpp, другая квантизация, нехватка памяти).")

props = requests.get(API + "/props").json()
N_CTX = props.get("default_generation_settings", {}).get("n_ctx", N_CTX)
print(f"\n✅ Сервер работает. Контекст: {N_CTX} токенов, KV-кэш: {KV_TYPE}")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 2. Настройки агента, GitHub и инструменты

In [ ]:
import os, re, json, time, subprocess, shutil, pathlib, inspect, functools, signal, requests

# ===== Репозиторий и режим работы =====
GITHUB_OWNER = "mlaa4ml"
GITHUB_REPO  = "tochki-multiplayer"
BASE_BRANCH  = None          # None = ветка репозитория по умолчанию
REPO_CLONE_URL = None        # None = https://github.com/OWNER/REPO.git (можно указать другой источник)
ISSUE_STATE  = "open"        # какие issues брать: open / closed / all
ONLY_ISSUES  = None          # например [12, 15]: решать только эти issues
ISSUE_LIMIT  = None          # например 3: взять только первые N (после сортировки по приоритету)
RERUN_DONE   = False         # True: заново решать issues, уже обработанные в прошлых запусках
MAX_TOTAL_SEC = None         # например 8*3600: не начинать новые issues после этого времени

# ===== Лимиты агента =====
STEPS_PER_ROUND   = 14       # вызовов инструментов за раунд
MAX_ROUNDS        = 4        # раундов на один issue (каждый раунд - новый чат)
TEXT_EXTRA_TURNS  = 6        # ходов сверх шагов: итоговый текст, ошибки формата
TOOL_OUTPUT_LIMIT = 12000    # символов результата инструмента, который видит модель
CMD_MAX_SEC       = 300      # потолок одной команды run_command
CMD_TOTAL_SEC     = 900      # суммарное время команд на один issue
LLM_MAX_OUTPUT_TOKENS = 16000
AGENT_TEMPERATURE = 0.6      # для вызовов инструментов ниже, чем в обычном чате
AGENT_THINKING    = True
STREAM_VIEW       = "dots"   # "dots" - точки по мере генерации, "text" - печатать текст, "off"
JUDGE             = True     # самопроверка решения той же моделью (ориентир, не истина)

# ===== Запись в GitHub (нужен токен: Issues: Read and write; для Codespaces: Codespaces: Read and write) =====
GITHUB_WRITE = True          # False: ничего не отправлять в GitHub, комментарии и новые issues сохраняются черновиками
COMMENT_STATUSES = ("solved", "partial")   # по каким итогам публиковать комментарий-отчёт (добавьте "no_action", если нужно)
MAX_COMMENTS_PER_ISSUE = 2   # комментарии-вопросы модели по ходу работы (итоговый отчёт публикует система и не считается)
MAX_ISSUES_CREATED_PER_RUN = 5
AGENT_LABEL = "qwen-agent"   # метка на issues, созданных агентом
PATCH_IN_COMMENT_LIMIT = 40000   # символов патча в комментарии (лимит GitHub на комментарий - 65536)

# ===== Codespaces: облачная среда для проверок, которые нельзя выполнить локально =====
USE_CODESPACES = True        # среда создаётся лениво, только при первом вызове run_in_codespace; в конце прогона останавливается
CODESPACE_MACHINE = "basicLinux32gb"
CODESPACE_IDLE_TIMEOUT_MIN = 30
CODESPACE_WORKDIR = f"/workspaces/{GITHUB_REPO}"
CODESPACE_CMD_MAX_SEC = 300
CODESPACE_TOTAL_SEC = 900    # суммарное время команд в Codespaces на один issue
CODESPACE_MAX_RUNS_PER_ISSUE = 6

# ===== Папки =====
REPO_DIR = "/tmp/agent_repo"
OUT_DIR  = "/kaggle/working/agent_out" if os.path.isdir("/kaggle/working") else "./agent_out"
os.makedirs(f"{OUT_DIR}/patches", exist_ok=True)
os.makedirs(f"{OUT_DIR}/traces", exist_ok=True)

# Токен нужен только для приватного репозитория или чтобы не упереться в лимит 60 запросов/час.
# Kaggle: Add-ons -> Secrets -> GITHUB_TOKEN (fine-grained, достаточно Contents: Read, Issues: Read).
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None
print("GITHUB_TOKEN:", "есть" if GITHUB_TOKEN else "нет (публичный репозиторий будет работать, но с лимитом запросов)")

In [ ]:
GITHUB_API = "https://api.github.com"
REPO_PATH  = f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
PRIORITY_ORDER = ["priority:p0", "priority:p1", "priority:p2"]
UNTRIAGED_RANK = 1.5

def _gh_headers():
    h = {"Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
    if GITHUB_TOKEN:
        h["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return h

def _gh_request(method, path, retries=3, **kwargs):
    last = None
    for attempt in range(retries):
        last = requests.request(method, GITHUB_API + path, headers=_gh_headers(), timeout=30, **kwargs)
        if last.status_code < 500 and last.status_code != 403:
            return last
        if last.status_code == 403 and "rate limit" not in last.text.lower():
            return last
        time.sleep(2 ** attempt)
    return last

def gh_paginate(path, params=None, max_pages=10):
    params = dict(params or {}); params.setdefault("per_page", 100)
    items, page = [], 1
    while page <= max_pages:
        params["page"] = page
        r = _gh_request("GET", path, params=params)
        if r.status_code != 200:
            raise RuntimeError(f"GitHub {r.status_code}: {r.text[:300]}")
        batch = r.json()
        items.extend(batch)
        if len(batch) < params["per_page"]:
            break
        page += 1
    return items

def _issue_rank(it):
    names = {l["name"] for l in it.get("labels", [])}
    for rank, label in enumerate(PRIORITY_ORDER):
        if label in names:
            return rank
    return UNTRIAGED_RANK

def fetch_issues(state="open"):
    """Все issues (без pull request), от важных к неважным, внутри приоритета - по номеру."""
    items = [it for it in gh_paginate(f"{REPO_PATH}/issues", {"state": state}) if "pull_request" not in it]
    items.sort(key=lambda it: (_issue_rank(it), it["number"]))
    return [{"number": it["number"], "title": it["title"], "body": it.get("body") or "",
             "labels": [l["name"] for l in it.get("labels", [])], "url": it["html_url"],
             "state": it["state"], "comments": it.get("comments", 0)} for it in items]

def issue_text(it, max_comments=30, comment_limit=3000):
    out = [f"ISSUE #{it['number']}: {it['title']}",
           f"state: {it['state']} | labels: {','.join(it['labels']) or '-'} | {it['url']}",
           "", "--- ТЕКСТ ---", it["body"] or "(пусто)"]
    if it.get("comments"):
        try:
            for c in gh_paginate(f"{REPO_PATH}/issues/{it['number']}/comments")[:max_comments]:
                out += ["", f"--- КОММЕНТАРИЙ {c['user']['login']} ---", (c.get("body") or "")[:comment_limit]]
        except Exception as e:
            out.append(f"\n(комментарии недоступны: {e})")
    return "\n".join(out)


# ================= запись: комментарии и новые issues =================
AGENT_MARKER = "<!-- QWEN_AGENT_REPORT -->"   # по нему находим собственный отчёт и обновляем, а не дублируем
NOTE_MARKER  = "<!-- QWEN_AGENT_NOTE -->"
GH_WRITE = {"ok": True, "reason": None}       # станет False после первого отказа по правам: дальше не пробуем

def _err(label, resp):
    return f"{label}: GitHub {resp.status_code}: {resp.text[:200]}"

def _write_failed(label, resp):
    msg = _err(label, resp)
    if resp.status_code in (401, 403, 404):    # у fine-grained токена нехватка прав часто приходит как 404
        GH_WRITE["ok"] = False
        GH_WRITE["reason"] = msg + " (похоже на нехватку прав токена: нужен Issues: Read and write). Дальше запись отключена."
    return msg

def gh_upsert_report(number, body):
    """Публикует итоговый отчёт; если отчёт агента в этом issue уже есть (маркер) - обновляет его. -> (url, ошибка)"""
    if not GH_WRITE["ok"]:
        return None, GH_WRITE["reason"]
    try:
        mine = next((c for c in gh_paginate(f"{REPO_PATH}/issues/{number}/comments")
                     if AGENT_MARKER in (c.get("body") or "")), None)
    except RuntimeError as e:
        return None, str(e)
    if mine:
        r = _gh_request("PATCH", f"{REPO_PATH}/issues/comments/{mine['id']}", json={"body": body}); want = 200
    else:
        r = _gh_request("POST", f"{REPO_PATH}/issues/{number}/comments", json={"body": body}); want = 201
    if r.status_code != want:
        return None, _write_failed("комментарий", r)
    return r.json().get("html_url"), None

def gh_post_note(number, body):
    if not GH_WRITE["ok"]:
        return None, GH_WRITE["reason"]
    r = _gh_request("POST", f"{REPO_PATH}/issues/{number}/comments", json={"body": body})
    if r.status_code != 201:
        return None, _write_failed("комментарий", r)
    return r.json().get("html_url"), None

def gh_create_issue(title, body, labels):
    if not GH_WRITE["ok"]:
        return None, GH_WRITE["reason"]
    r = _gh_request("POST", f"{REPO_PATH}/issues", json={"title": title, "body": body, "labels": labels})
    if r.status_code != 201:
        return None, _write_failed("создание issue", r)
    return r.json(), None

# Названия ВСЕХ issues (и закрытых) - чтобы агент не заводил дубликаты
KNOWN_ISSUES = {}
def load_known_issues():
    KNOWN_ISSUES.clear()
    for it in gh_paginate(f"{REPO_PATH}/issues", {"state": "all"}):
        if "pull_request" not in it:
            KNOWN_ISSUES[it["number"]] = it["title"]

def _title_tokens(s):
    return set(re.findall(r"\w{3,}", (s or "").lower()))

def find_duplicate(title, threshold=0.6):
    t = _title_tokens(title)
    for n, other in KNOWN_ISSUES.items():
        o = _title_tokens(other)
        if t and o and len(t & o) / len(t | o) >= threshold:
            return n, other
    return None

In [ ]:
class ContextExceeded(Exception):
    pass

def _ts():
    return time.strftime("%H:%M:%S")

def llm_stream(messages, max_tokens=None, think=None, show=True):
    """Один запрос к llama-server со стримингом. Возвращает {content, finish, usage}. Размышления отбрасываются."""
    think = AGENT_THINKING if think is None else think
    payload = {"model": "qwen", "messages": messages, "stream": True,
               "stream_options": {"include_usage": True},
               "max_tokens": max_tokens or LLM_MAX_OUTPUT_TOKENS,
               "temperature": AGENT_TEMPERATURE if think else 0.7,
               "top_p": 0.95 if think else 0.8, "top_k": 20, "min_p": 0.0,
               "chat_template_kwargs": {"enable_thinking": bool(think)}}
    r = requests.post(API + "/v1/chat/completions", json=payload, stream=True, timeout=(10, 3600))
    r.encoding = "utf-8"
    if r.status_code != 200:
        body = r.text
        if r.status_code == 400 and re.search(r"context|exceed", body, re.I):
            raise ContextExceeded(body[:300])
        raise RuntimeError(f"HTTP {r.status_code}: {body[:500]}")
    content, usage, finish, chunks, t0 = [], None, None, 0, time.time()
    for raw in r.iter_lines(decode_unicode=True):
        if not raw or not raw.startswith("data:"):
            continue
        data = raw[5:].strip()
        if data == "[DONE]":
            break
        try:
            obj = json.loads(data)
        except json.JSONDecodeError:
            continue
        usage = obj.get("usage") or usage
        for ch in obj.get("choices") or []:
            d = ch.get("delta") or {}
            c = d.get("content")
            if c:
                content.append(c)
                if show and STREAM_VIEW == "text":
                    print(c, end="", flush=True)
            if d.get("reasoning_content") or c:
                chunks += 1
                if show and STREAM_VIEW == "dots" and chunks % 60 == 0:
                    print(".", end="", flush=True)
            if ch.get("finish_reason"):
                finish = ch["finish_reason"]
    if show and STREAM_VIEW != "off":
        n = (usage or {}).get("completion_tokens")
        print(f" ← модель: {n or chunks} ток за {time.time() - t0:.0f} с", flush=True)
    return {"content": "".join(content), "finish": finish, "usage": usage or {}}

def clean_answer(text):
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S)
    if "</think>" in text:
        text = text.split("</think>", 1)[1]
    return text.strip()

# ---------------- бюджет шагов ----------------
PRODUCTIVE_TOOLS = {"write_file", "patch_file", "delete_file", "commit", "report_work", "report_no_action_needed"}

class StepBudget:
    def __init__(self, limit, log=None):
        self.limit, self.used, self.made_progress = limit, 0, False
        self.log = log if log is not None else []
    def consume(self):
        if self.used >= self.limit:
            return False
        self.used += 1
        return True
    @property
    def remaining(self):
        return max(0, self.limit - self.used)

def _step_suffix(b):
    s = f"\n[шаг {b.used}/{b.limit}]"
    if b.remaining == 0:
        s += " бюджет раунда исчерпан: заверши раунд коротким текстом (что сделано, что осталось)."
    elif b.remaining <= 2:
        s += (f" осталось {b.remaining}. Не отчитывайся, пока не сделано ВСЁ: если сделано всё — "
              "report_work(..., final=True); если нет — заверши раунд текстом (что сделано, что осталось).")
    elif b.remaining <= 4 and not b.made_progress:
        s += (f" осталось {b.remaining}, а правок в этом раунде ещё не было — только чтение. "
              "Не перечитывай известное: переходи к правке.")
    return s

def _short(args, kwargs, limit=150):
    s = ", ".join([repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()])
    return s if len(s) <= limit else s[:limit] + "…"

def with_budget(fn, budget):
    @functools.wraps(fn)
    def wrapped(*args, **kwargs):
        if not budget.consume():
            return f"ERROR: бюджет шагов раунда исчерпан ({budget.limit}). Заверши раунд текстом."
        t0 = time.time()
        print(f"[{_ts()}] → {fn.__name__}({_short(args, kwargs)})", flush=True)
        try:
            result = fn(*args, **kwargs)
        except Exception as e:
            result = f"ERROR: {type(e).__name__}: {e}"
        print(f"[{_ts()}] ← {fn.__name__} за {time.time() - t0:.0f} с", flush=True)
        if fn.__name__ in PRODUCTIVE_TOOLS and not str(result).startswith("ERROR"):
            budget.made_progress = True
        budget.log.append({"tool": fn.__name__, "args": kwargs or list(args), "result": str(result)[:1500]})
        return f"{result}{_step_suffix(budget)}"
    return wrapped

# ---------------- текстовый протокол вызова инструментов ----------------
_CALL_RE = re.compile(r"^\s*@@CALL[ \t]+([A-Za-z_][A-Za-z0-9_]*)\s*$")
_ARG_RE  = re.compile(r"^\s*@@ARG[ \t]+([A-Za-z_][A-Za-z0-9_]*)[ \t]*(?::[ \t]?(.*))?$")
_END_RE  = re.compile(r"^\s*@@END\s*$")
_STRIP_CALL_RE = re.compile(r"(?ms)^\s*@@CALL\b.*?(?:^\s*@@END\s*$|\Z)")
_TRAILING_NEWLINE_ARGS = {("write_file", "content")}

def text_protocol_prompt():
    return (
        "ФОРМАТ ВЫЗОВА ИНСТРУМЕНТОВ. Нативных вызовов функций здесь нет: чтобы вызвать инструмент, "
        "напиши в ответе блок РОВНО такого вида:\n\n"
        "@@CALL имя_инструмента\n@@ARG имя_аргумента\nзначение (может занимать несколько строк — как есть, "
        "без кавычек и экранирования)\n@@ARG другой_аргумент\nзначение\n@@END\n\n"
        "Пример:\n@@CALL read_file\n@@ARG path\nsrc/app.js\n@@ARG start_line\n1\n@@END\n\n"
        "Правила:\n"
        "- ОДИН вызов за сообщение. Всё после @@END игнорируется; результат придёт следующим сообщением "
        "(«РЕЗУЛЬТАТ ...»). Не придумывай результаты сам.\n"
        "- Ответ БЕЗ блока @@CALL означает, что ты закончил раунд. Не пиши «сейчас прочитаю...» без блока: "
        "если хочешь действовать — сразу вызывай инструмент.\n"
        "- Не оборачивай блок в ``` и ничего не добавляй между строкой @@ARG и значением. Значение — все строки "
        "до следующей строки @@ARG или @@END. Необязательные аргументы можно опустить. Булевы значения — true/false.\n"
        f"- Длина одного ответа ограничена (около {LLM_MAX_OUTPUT_TOKENS} токенов вместе с размышлениями). "
        "Блок без @@END (оборванный ответ) НЕ выполняется. Небольшие правки делай через patch_file, а не "
        "переписывай файл целиком.\n\nИНСТРУМЕНТЫ:\n"
    )

def _ann_name(a):
    return "str" if a is inspect.Parameter.empty else getattr(a, "__name__", str(a))

def tool_signature(t):
    parts = []
    for p in inspect.signature(t).parameters.values():
        s = f"{p.name}: {_ann_name(p.annotation)}"
        if p.default is not inspect.Parameter.empty:
            s += f" = {p.default!r}"
        parts.append(s)
    return f"{t.__name__}({', '.join(parts)})"

def build_tool_spec(tools):
    return "\n".join(f"- {tool_signature(t)} — {' '.join((inspect.getdoc(t) or '').split())}" for t in tools)

def parse_text_call(text):
    """(name, raw_args, error, n_calls). name=None и error=None: вызова нет."""
    lines = (text or "").replace("\r\n", "\n").split("\n")
    idxs = [i for i, l in enumerate(lines) if _CALL_RE.match(l)]
    if not idxs:
        return None, None, None, 0
    name = _CALL_RE.match(lines[idxs[0]]).group(1)
    args, cur, buf, closed = {}, None, [], False
    for l in lines[idxs[0] + 1:]:
        if _END_RE.match(l):
            closed = True
            break
        m = _ARG_RE.match(l)
        if m:
            if cur is not None:
                args[cur] = "\n".join(buf)
            cur, buf = m.group(1), ([m.group(2)] if m.group(2) is not None else [])
        elif cur is not None:
            buf.append(l)
    if cur is not None:
        args[cur] = "\n".join(buf)
    if not closed:
        return name, args, ("вызов не закончен: нет строки @@END (ответ мог быть оборван лимитом длины — "
                            "для больших правок используй patch_file)"), len(idxs)
    return name, args, None, len(idxs)

def coerce_args(t, name, raw_args):
    sig = inspect.signature(t)
    kwargs = {}
    for k, raw in raw_args.items():
        p = sig.parameters.get(k)
        if p is None:
            raise ValueError(f"неизвестный аргумент «{k}»")
        ann, has_default = p.annotation, p.default is not inspect.Parameter.empty
        if ann in (int, "int", float, "float", bool, "bool"):
            s = raw.strip()
            if s == "" and has_default:
                continue
            try:
                if ann in (int, "int"):
                    kwargs[k] = int(s)
                elif ann in (float, "float"):
                    kwargs[k] = float(s)
                elif s.lower() in ("true", "1", "yes", "да"):
                    kwargs[k] = True
                elif s.lower() in ("false", "0", "no", "нет"):
                    kwargs[k] = False
                else:
                    raise ValueError(s)
            except ValueError:
                raise ValueError(f"аргумент «{k}» должен быть {_ann_name(ann)}, получено {s[:40]!r}") from None
        else:
            v = raw if "\n" in raw else raw.strip()
            if (name, k) in _TRAILING_NEWLINE_ARGS and v and not v.endswith("\n"):
                v += "\n"
            kwargs[k] = v
    try:
        sig.bind(**kwargs)
    except TypeError as e:
        raise ValueError(str(e)) from None
    return kwargs

def strip_calls(text):
    return _STRIP_CALL_RE.sub("", text or "").strip()

In [ ]:
# Codespaces: облачная среда для проверок. Код у нас только локально (push запрещён), поэтому перед КАЖДОЙ командой
# локальный diff переносится в чистое состояние базовой ветки внутри Codespace. Жизненный цикл - через REST, команды - через gh CLI.
import base64, shlex, hashlib

CS = {"name": None, "error": None, "gh_ready": None, "used": False}
SSH_NOT_READY_MARKERS = ("error getting ssh server details", "failed to start ssh server")
SSH_READY_RETRIES, SSH_READY_RETRY_DELAY_SEC = 6, 10
_TIMEOUT_EXIT_CODES = (124, 137)

def _ensure_gh_cli():
    """Ставит gh CLI (если нет) и проверяет авторизацию по GITHUB_TOKEN. Один раз за прогон."""
    if CS["gh_ready"] is not None:
        return CS["gh_ready"]
    if shutil.which("gh") is None:
        steps = ["apt-get update -qq", "apt-get install -y -qq curl",
                 "curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg -o /usr/share/keyrings/githubcli-archive-keyring.gpg",
                 "chmod go+r /usr/share/keyrings/githubcli-archive-keyring.gpg",
                 "bash -c 'echo \"deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] "
                 "https://cli.github.com/packages stable main\" > /etc/apt/sources.list.d/github-cli.list'",
                 "apt-get update -qq", "apt-get install -y -qq gh"]
        for cmd in steps:
            r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            if r.returncode != 0:
                print(f"gh CLI: шаг установки провалился ({cmd[:40]}...): {r.stderr[:200]}")
                CS["gh_ready"] = False
                return False
    os.environ["GH_TOKEN"] = GITHUB_TOKEN or ""
    check = subprocess.run(["gh", "auth", "status"], capture_output=True, text=True)
    CS["gh_ready"] = check.returncode == 0
    print("gh CLI готов и авторизован." if CS["gh_ready"] else (check.stdout + check.stderr)[:300])
    return CS["gh_ready"]

def gh_create_codespace(branch):
    r = _gh_request("POST", f"{REPO_PATH}/codespaces",
                    json={"ref": branch, "machine": CODESPACE_MACHINE, "idle_timeout_minutes": CODESPACE_IDLE_TIMEOUT_MIN})
    if r.status_code not in (200, 201):
        return None, _err("создание codespace", r) + " (нужен токен с Codespaces: Read and write)"
    return r.json(), None

def gh_wait_codespace_available(name, timeout_sec=240):
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        r = _gh_request("GET", f"/user/codespaces/{name}")
        state = (r.json() if r.status_code == 200 else {}).get("state")
        if state == "Available":
            return True, state
        if state in ("Failed", "Deleted", "Shutdown", "Unavailable"):
            return False, state
        time.sleep(5)
    return False, "Timeout"

def gh_find_codespace_for_branch(branch):
    r = _gh_request("GET", f"{REPO_PATH}/codespaces")
    if r.status_code != 200:
        return None
    for cs in r.json().get("codespaces", []):
        if cs.get("ref") == branch and cs.get("state") not in ("Shutdown", "Deleted", "Failed"):
            return cs["name"]
    return None

def ensure_codespace(base):
    if CS["name"]:
        return CS["name"], None
    if CS["error"]:
        return None, CS["error"]
    if not GITHUB_TOKEN or not _ensure_gh_cli():
        CS["error"] = "gh CLI недоступен или токен не авторизован: Codespaces не работают, проверяй локально через run_command"
        return None, CS["error"]
    name = gh_find_codespace_for_branch(base)
    if not name:
        print(f"[{_ts()}] создаю Codespace на ветке {base} (1-3 минуты)…", flush=True)
        data, err = gh_create_codespace(base)
        if err:
            CS["error"] = err
            return None, err
        name = data["name"]
    ok, state = gh_wait_codespace_available(name)
    if not ok:
        CS["error"] = f"Codespace не стал доступен (состояние: {state})"
        return None, CS["error"]
    CS["name"], CS["used"] = name, True
    print(f"[{_ts()}] Codespace готов: {name}", flush=True)
    return name, None

def stop_codespace_if_any():
    if CS["name"]:
        r = _gh_request("POST", f"/user/codespaces/{CS['name']}/stop")
        print(f"Codespace {CS['name']} " + ("остановлен." if r.status_code in (200, 202) else f"не удалось остановить ({r.status_code}), его остановит idle_timeout"))
        CS["name"] = None

def _codespace_script(command, patch_b64, remote_limit):
    """Однострочный удалённый скрипт: чистая база -> перенос локального diff -> команда с таймаутом на стороне Codespace."""
    q = shlex.quote(command)
    return "; ".join([
        f'cd {shlex.quote(CODESPACE_WORKDIR)} 2>/dev/null || cd "$(ls -d /workspaces/*/ 2>/dev/null | head -1)"',
        "git reset -q --hard HEAD 2>/dev/null; git clean -fdq 2>/dev/null",
        f"echo {patch_b64} | base64 -d > /tmp/agent.patch",
        'if [ -s /tmp/agent.patch ]; then git apply --whitespace=nowarn /tmp/agent.patch || echo "[sync] ОШИБКА git apply: правки не перенесены" >&2; fi',
        f"if command -v timeout >/dev/null 2>&1; then timeout -k 5 {int(remote_limit)} bash -c {q}; else bash -c {q}; fi",
    ])

def _codespace_run(name, script, timeout_sec):
    # ssh склеивает argv пробелами и отдаёт удалённому шеллу одной строкой, поэтому без shlex.quote команда ломается
    return subprocess.run(["gh", "codespace", "ssh", "--codespace", name, "--", "bash", "-lc", shlex.quote(script)],
                          capture_output=True, text=True, timeout=timeout_sec)

def codespace_exec(sess, command, timeout_sec=120):
    base = sess["base"]
    if sess["cs_runs"] >= CODESPACE_MAX_RUNS_PER_ISSUE:
        return f"ERROR: лимит запусков в Codespaces на issue исчерпан ({CODESPACE_MAX_RUNS_PER_ISSUE}). Опиши в отчёте, что осталось проверить."
    left = CODESPACE_TOTAL_SEC - sess["cs_sec"]
    if left <= 0:
        return f"ERROR: исчерпан общий лимит времени Codespaces на issue ({CODESPACE_TOTAL_SEC} с)."
    git("add", "-A", "-N")
    patch = git("diff", "--binary", base)
    key = (" ".join(command.split()), hashlib.sha1(patch.encode()).hexdigest())
    if key in sess["cs_cache"]:
        return "[кэш: код не менялся с прошлого запуска этой команды, повторный запуск не нужен]\n" + sess["cs_cache"][key]
    b64 = base64.b64encode(patch.encode()).decode()
    if len(b64) > 90000:
        return "ERROR: патч слишком велик для переноса в Codespace. Проверь локально через run_command или сократи изменения."
    name, err = ensure_codespace(base)
    if err:
        return "ERROR: " + err
    to = max(10, min(int(timeout_sec), CODESPACE_CMD_MAX_SEC, left))
    t0, result = time.time(), None
    for attempt in range(1, SSH_READY_RETRIES + 1):
        try:
            r = _codespace_run(name, _codespace_script(command, b64, to), to + 90)
        except subprocess.TimeoutExpired:
            result = f"ERROR: команда не уложилась в {to} с и остановлена. Уменьши объём проверки."
            break
        if r.returncode != 0 and any(m in r.stderr.lower() for m in SSH_NOT_READY_MARKERS) and attempt < SSH_READY_RETRIES:
            time.sleep(SSH_READY_RETRY_DELAY_SEC)
            continue
        out = f"exit={r.returncode}\n--- stdout ---\n{r.stdout}\n--- stderr ---\n{r.stderr}"
        if len(out) > 6000:
            out = "[... начало обрезано ...]\n" + out[-6000:]
        if r.returncode in _TIMEOUT_EXIT_CODES:
            out += f"\n[команда остановлена по лимиту {to} с]"
        result = out
        sess["commands"].append({"cmd": "[codespace] " + command[:180], "exit": r.returncode})
        break
    sess["cs_runs"] += 1; sess["cs_total"] += 1; sess["cs_sec"] += time.time() - t0
    if result and not result.startswith("ERROR"):
        sess["cs_cache"][key] = result
    return result or "ERROR: sshd в Codespace не поднялся (проверьте .devcontainer: нужна фича ghcr.io/devcontainers/features/sshd:1)"

In [ ]:
# ---------------- git и безопасность путей ----------------
def git(*args, check=True, timeout=180):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_AUTHOR_NAME": "qwen-agent",
           "GIT_AUTHOR_EMAIL": "agent@localhost", "GIT_COMMITTER_NAME": "qwen-agent",
           "GIT_COMMITTER_EMAIL": "agent@localhost"}
    r = subprocess.run(["git", *args], cwd=REPO_DIR, capture_output=True, text=True, timeout=timeout, env=env)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} завершился с ошибкой: {(r.stderr or r.stdout).strip()[:500]}")
    return r.stdout

def clone_repo():
    """Свежий клон репозитория. Токен нужен только для клонирования и в рабочей копии не остаётся."""
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    clean_url = REPO_CLONE_URL or f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
    url = clean_url
    if GITHUB_TOKEN and url.startswith("https://github.com/"):
        url = url.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1)
    cmd = ["git", "clone", "--depth", "100"] + (["--branch", BASE_BRANCH] if BASE_BRANCH else []) + [url, REPO_DIR]
    r = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode != 0:
        raise RuntimeError("git clone не удался: " + r.stderr.replace(GITHUB_TOKEN or "\0", "***")[:500])
    git("remote", "set-url", "origin", clean_url)
    with open(f"{REPO_DIR}/.git/info/exclude", "a") as f:        # мусор от запуска тестов не попадёт в коммиты
        f.write("\n__pycache__/\n*.pyc\n.pytest_cache/\nnode_modules/\n.venv/\n")
    return BASE_BRANCH or git("rev-parse", "--abbrev-ref", "HEAD").strip()

def prepare_branch(issue_number, base):
    git("checkout", "-f", base)
    git("clean", "-fd")
    branch = f"agent/issue-{issue_number}"
    git("checkout", "-B", branch)
    return branch

def _safe(path):
    root = pathlib.Path(REPO_DIR).resolve()
    p = (root / path).resolve()
    if p != root and root not in p.parents:
        raise ValueError("путь вне репозитория")
    if ".git" in p.relative_to(root).parts:
        raise ValueError("нельзя трогать .git")
    return p

def _clip(text, limit=None):
    text, limit = str(text), limit or TOOL_OUTPUT_LIMIT
    return text if len(text) <= limit else text[:limit] + f"\n...[обрезано, всего {len(text)} символов]"

def _has_changes(base):
    git("add", "-A", "-N")
    return bool(git("diff", base).strip())

def _ahead(base):
    return int(git("rev-list", "--count", f"{base}..HEAD").strip() or 0)

BLOCKED_CMDS = [
    (r"\bgit\s+(?:-\S+\s+)*(push|remote|config|credential)\b", "git push/remote/config запрещены: изменения сохраняет система"),
    (r"\bsudo\b", "sudo запрещён"),
    (r"\brm\s+-[a-zA-Z]*[rf][a-zA-Z]*\s+(/|~|\$HOME|\.\.)(\s|$)", "опасное удаление запрещено"),
    (r"curl[^|;&]*\|\s*(ba)?sh", "запуск скриптов из сети запрещён"),
    (r"\bgh\s", "gh CLI недоступен"),
    (r"(^|\s)(shutdown|reboot|mkfs)\b|\bdd\s+if=", "системные команды запрещены"),
    (r"GITHUB_TOKEN|KAGGLE_KEY|\.git/config", "доступ к секретам запрещён"),
]
_SECRET_ENV = re.compile(r"TOKEN|SECRET|KEY|PASSWORD|KAGGLE", re.I)

CREATED_ISSUES = []          # все issues, созданные агентом (и черновики); сохраняются в OUT_DIR/created_issues.json
RUN_CREATED = {"n": 0}       # сколько создано в ЭТОМ прогоне (лимит MAX_ISSUES_CREATED_PER_RUN)
CREATED_PATH = f"{OUT_DIR}/created_issues.json"
if os.path.exists(CREATED_PATH):
    CREATED_ISSUES.extend(json.load(open(CREATED_PATH, encoding="utf-8")))

def _save_created():
    json.dump(CREATED_ISSUES, open(CREATED_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

def _draft(name, text):
    os.makedirs(f"{OUT_DIR}/drafts", exist_ok=True)
    pathlib.Path(f"{OUT_DIR}/drafts/{name}").write_text(text, encoding="utf-8")

def new_sess(issue, base, branch, text):
    return {"issue": issue, "base": base, "branch": branch, "issue_text": text, "final": None, "no_action": None,
            "reports": [], "commands": [], "cmd_sec": 0.0, "cs_runs": 0, "cs_total": 0, "cs_sec": 0.0, "cs_cache": {},
            "comments": 0, "created": []}

def make_tools(sess, budget):
    """Инструменты, замкнутые на один issue (sess) и бюджет раунда."""
    base = sess["base"]

    def get_issue() -> str:
        """Показать текст текущего issue целиком (с комментариями)."""
        return _clip(sess["issue_text"], 30000)

    def list_files(subdir: str = "") -> str:
        """Список файлов репозитория (путь относительно корня). subdir — показать только эту папку."""
        files = [f for f in git("ls-files", "-co", "--exclude-standard").splitlines()
                 if not subdir or f.startswith(subdir.strip("/") + "/") or f == subdir]
        if not files:
            return "Файлов не найдено."
        head = f"Файлов: {len(files)}" + (" (показаны первые 400, уточни subdir)" if len(files) > 400 else "")
        return head + "\n" + "\n".join(files[:400])

    def read_file(path: str, start_line: int = 1, max_lines: int = 250) -> str:
        """Прочитать файл (текст без номеров строк). Большие файлы читай частями: start_line и max_lines."""
        p = _safe(path)
        if not p.is_file():
            return f"ERROR: файла нет: {path}"
        data = p.read_bytes()
        if b"\0" in data[:4096]:
            return f"ERROR: {path} — бинарный файл ({len(data)} байт)"
        lines = data.decode("utf-8", "replace").splitlines()
        a = max(1, start_line); b = min(len(lines), a + max(1, max_lines) - 1)
        body = "\n".join(lines[a - 1:b])
        tail = f"\n[... ещё {len(lines) - b} строк: читай с start_line={b + 1}]" if b < len(lines) else ""
        return _clip(f"[{path}: строки {a}-{b} из {len(lines)}]\n{body}{tail}")

    def search_code(query: str, subdir: str = "") -> str:
        """Поиск подстроки (без учёта регистра) по файлам репозитория. Результат: путь:строка: текст."""
        q, hits = query.lower(), []
        for f in git("ls-files", "-co", "--exclude-standard").splitlines():
            if subdir and not f.startswith(subdir.strip("/")):
                continue
            p = pathlib.Path(REPO_DIR) / f
            try:
                if not p.is_file() or p.stat().st_size > 1_000_000:
                    continue
                for i, line in enumerate(p.read_text("utf-8", "ignore").splitlines(), 1):
                    if q in line.lower():
                        hits.append(f"{f}:{i}: {line.strip()[:200]}")
                        if len(hits) >= 60:
                            return "\n".join(hits) + "\n[показаны первые 60 совпадений, уточни запрос]"
            except OSError:
                continue
        return "\n".join(hits) or "Совпадений нет."

    def write_file(path: str, content: str) -> str:
        """Создать файл или полностью перезаписать его (нужен ПОЛНЫЙ текст). Для небольших правок используй patch_file."""
        p = _safe(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding="utf-8")
        warn = ""
        if path.endswith(".py"):
            try:
                compile(content, path, "exec")
            except SyntaxError as e:
                warn = f"\nВНИМАНИЕ: файл не компилируется: {e}"
        elif path.endswith(".json"):
            try:
                json.loads(content)
            except ValueError as e:
                warn = f"\nВНИМАНИЕ: невалидный JSON: {e}"
        return f"OK: записан {path} ({content.count(chr(10)) + 1} строк){warn}"

    def patch_file(path: str, old_text: str, new_text: str) -> str:
        """Заменить фрагмент old_text на new_text. old_text должен совпасть символ в символ и встречаться ровно один раз."""
        p = _safe(path)
        if not p.is_file():
            return f"ERROR: файла нет: {path}"
        src = p.read_text("utf-8")
        n = src.count(old_text)
        if not old_text:
            return "ERROR: old_text пуст"
        if n == 0:
            return "ERROR: old_text не найден (проверь пробелы и отступы: прочитай нужный фрагмент через read_file)"
        if n > 1:
            return f"ERROR: old_text встречается {n} раз: добавь соседние строки, чтобы фрагмент был уникальным"
        p.write_text(src.replace(old_text, new_text, 1), encoding="utf-8")
        return f"OK: заменён фрагмент в {path}"

    def delete_file(path: str) -> str:
        """Удалить файл."""
        p = _safe(path)
        if not p.is_file():
            return f"ERROR: файла нет: {path}"
        p.unlink()
        return f"OK: удалён {path}"

    def run_command(command: str, timeout_sec: int = 120) -> str:
        """Выполнить shell-команду в корне репозитория (тесты, линтер, запуск скрипта). Без интерактива и вечных процессов."""
        for pat, why in BLOCKED_CMDS:
            if re.search(pat, command):
                return f"ERROR: команда заблокирована: {why}"
        left = CMD_TOTAL_SEC - sess["cmd_sec"]
        if left <= 0:
            return f"ERROR: исчерпан общий лимит времени команд на этот issue ({CMD_TOTAL_SEC} с). Опиши в отчёте, что осталось проверить."
        to = max(5, min(int(timeout_sec), CMD_MAX_SEC, left))
        env = {k: v for k, v in os.environ.items() if not _SECRET_ENV.search(k)}
        env["PYTHONDONTWRITEBYTECODE"] = "1"   # иначе правка той же длины в ту же секунду даёт устаревший .pyc
        t0 = time.time()
        proc = subprocess.Popen(["bash", "-lc", command], cwd=REPO_DIR, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, env=env, start_new_session=True)
        try:
            out, _ = proc.communicate(timeout=to)
            code = proc.returncode
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            out, _ = proc.communicate()
            code = "TIMEOUT"
            out = (out or "") + f"\n[команда остановлена по таймауту {to} с]"
        sess["cmd_sec"] += time.time() - t0
        sess["commands"].append({"cmd": command[:200], "exit": code})
        out = out or ""
        if len(out) > 6000:
            out = "[... начало обрезано ...]\n" + out[-6000:]
        return f"exit={code}\n{out}"

    def git_diff() -> str:
        """Показать все изменения в рабочей копии относительно базовой ветки (и закоммиченные, и нет)."""
        git("add", "-A", "-N")
        d = git("diff", base)
        return _clip(d, 20000) if d.strip() else "Изменений относительно базовой ветки нет."

    def commit(message: str) -> str:
        """Закоммитить все текущие изменения в рабочую ветку (сообщение коммита — по-английски или по-русски, кратко)."""
        git("add", "-A")
        if not git("diff", "--cached", "--name-only").strip():
            return "ERROR: нечего коммитить"
        git("commit", "-m", message[:200])
        return f"OK: коммит {git('rev-parse', '--short', 'HEAD').strip()} ({', '.join(git('diff', '--name-only', 'HEAD~1', 'HEAD').split()[:8])})"

    def report_work(summary: str, final: bool = False) -> str:
        """Отчёт по issue. Без final — промежуточная заметка, задачу НЕ засчитывает. final=True — ОДИН раз, когда сделано ВСЁ: summary с разделами «Сделано», «Не сделано / ограничения», «Проверка»."""
        if not final:
            sess["reports"].append(summary)
            return "OK: промежуточный отчёт записан (задачу не засчитывает)."
        if not _has_changes(base):
            return ("ERROR: в рабочей копии нет изменений относительно базы — финальный отчёт не принят. "
                    "Внеси правку или используй report_no_action_needed.")
        if sess["final"] and not git("status", "--porcelain").strip():
            return "ERROR: финальный отчёт уже принят, новых изменений нет."
        if git("status", "--porcelain").strip():
            git("add", "-A")
            git("commit", "-m", f"agent: issue #{sess['issue']}")
        sess["final"] = summary
        return f"OK: финальный отчёт принят (коммитов в ветке: {_ahead(base)}). Раунд можно завершать."

    def report_no_action_needed(reason: str) -> str:
        """Использовать ВМЕСТО report_work, если после честного изучения issue и кода правка репозитория НЕ нужна. reason — конкретная причина."""
        if _has_changes(base):
            return "ERROR: в рабочей копии есть изменения. Если они нужны — report_work(final=True)."
        sess["no_action"] = reason
        return "OK: принято. Причина будет проверена."

    def comment_issue(body: str) -> str:
        """Оставить комментарий в ТЕКУЩЕМ issue: вопрос людям или пояснение по ходу работы. Итоговый отчёт публикуется автоматически, его дублировать не нужно. Лимит комментариев на issue."""
        if sess["comments"] >= MAX_COMMENTS_PER_ISSUE:
            return f"ERROR: лимит комментариев на issue исчерпан ({MAX_COMMENTS_PER_ISSUE})."
        if len(body.strip()) < 20:
            return "ERROR: комментарий слишком короткий: напиши конкретный вопрос или пояснение."
        text = f"{NOTE_MARKER}\n{body.strip()[:6000]}\n\n— 🤖 локальная модель (вопрос/пояснение по ходу работы)"
        sess["comments"] += 1
        _draft(f"note-issue-{sess['issue']}-{sess['comments']}.md", text)
        if not GITHUB_WRITE:
            return "OK: черновик комментария сохранён (GITHUB_WRITE=False), в GitHub не отправлен."
        url, err = gh_post_note(sess["issue"], text)
        return f"OK: комментарий опубликован: {url}" if not err else f"ERROR: не удалось опубликовать: {err}"

    def create_issue(title: str, body: str, priority: str = "p2") -> str:
        """Завести НОВЫЙ issue под отдельную проблему, найденную вне рамок текущей задачи (не чини её молча и не раздувай текущий diff). priority: p0 — блокирует использование репозитория, p1 — реальная проблема с обходным путём, p2 — мелочь. В body: что не так, где (файл/строка), как воспроизвести, как предлагаешь исправить. Дубликаты отклоняются, лимит за прогон."""
        p = priority.strip().lower().replace("priority:", "")
        if p not in ("p0", "p1", "p2"):
            return "ERROR: priority должен быть p0, p1 или p2"
        if RUN_CREATED["n"] >= MAX_ISSUES_CREATED_PER_RUN:
            return f"ERROR: лимит новых issues за прогон исчерпан ({MAX_ISSUES_CREATED_PER_RUN}). Опиши находку в итоговом отчёте."
        if len(title.strip()) < 8 or len(body.strip()) < 40:
            return "ERROR: нужны содержательные title (от 8 символов) и body (от 40): что не так, где и как воспроизвести."
        dup = find_duplicate(title)
        if dup:
            return f"ERROR: похожий issue уже есть: #{dup[0]} «{dup[1][:80]}». Новый не создан."
        full = body.strip()[:8000] + f"\n\n---\n🤖 Найдено автоматически при работе над #{sess['issue']} (локальная модель). {NOTE_MARKER}"
        rec = {"found_in": sess["issue"], "title": title.strip()[:200], "priority": p, "number": None, "url": None, "draft": not GITHUB_WRITE}
        if GITHUB_WRITE:
            issue, err = gh_create_issue(rec["title"], full, [f"priority:{p}", AGENT_LABEL])
            if err:
                return f"ERROR: issue не создан: {err}"
            rec["number"], rec["url"] = issue["number"], issue["html_url"]
            KNOWN_ISSUES[issue["number"]] = rec["title"]
        else:
            KNOWN_ISSUES[-(len(CREATED_ISSUES) + 1)] = rec["title"]
            _draft(f"new-issue-{len(CREATED_ISSUES) + 1}.md", f"# {rec['title']}\n\nlabels: priority:{p}, {AGENT_LABEL}\n\n{full}")
        CREATED_ISSUES.append(rec); sess["created"].append(rec); RUN_CREATED["n"] += 1
        _save_created()
        return (f"OK: создан issue #{rec['number']}: {rec['url']}" if GITHUB_WRITE else
                "OK: черновик нового issue сохранён (GITHUB_WRITE=False), в GitHub не отправлен.")

    def run_in_codespace(command: str, timeout_sec: int = 120) -> str:
        """Выполнить команду в облачной среде GitHub Codespaces с ТВОИМИ правками (они переносятся из локальной копии автоматически, push не нужен). Используй, когда локально проверить нельзя: нужно окружение/зависимости из devcontainer репозитория, другая версия рантайма, база данных и т. п. Первый вызов создаёт среду (1-3 минуты). Остальное проверяй локально через run_command."""
        return codespace_exec(sess, command, timeout_sec)

    tools = [get_issue, list_files, read_file, search_code, write_file, patch_file, delete_file,
             run_command] + ([run_in_codespace] if (USE_CODESPACES and GITHUB_TOKEN) else []) + \
            [git_diff, commit, comment_issue, create_issue, report_work, report_no_action_needed]
    return [with_budget(t, budget) for t in tools]

## 3. Логика агента: раунды, отчёт

In [ ]:
SYSTEM_PROMPT = (
    f"Ты — инженер-агент. Ты работаешь с локальной копией git-репозитория {GITHUB_OWNER}/{GITHUB_REPO} "
    "ТОЛЬКО через инструменты. Рабочая ветка для задачи уже создана и выбрана. Твоя задача — решить ОДИН issue.\n\n"
    "Цикл работы: изучить issue -> найти код (list_files / search_code / read_file) -> внести правку "
    "(patch_file для небольших изменений, write_file для новых файлов) -> проверить (run_command: тесты, запуск "
    "скрипта, если они есть) -> git_diff -> report_work(final=True).\n\n"
    "Правила:\n"
    "- Никогда не выдумывай содержимое файлов: сначала читай. Перед patch_file прочитай нужный фрагмент: old_text "
    "должен совпасть символ в символ и встречаться один раз.\n"
    "- git push и pull request недоступны: изменения остаются локально. Система сама сохранит diff твоей ветки и "
    "опубликует итоговый отчёт с патчем комментарием в issue. Коммитить можно через commit(message) или "
    "автоматически при финальном отчёте.\n"
    "- comment_issue — только чтобы задать людям вопрос или оставить пояснение по ходу работы; итоговый отчёт "
    "дублировать не нужно.\n"
    "- Если находишь ОТДЕЛЬНУЮ проблему вне своей задачи (баг, уязвимость, сломанный тест) — заведи под неё "
    "create_issue(title, body, priority) и продолжай свою задачу: не чини её молча и не раздувай свой diff. "
    "В body: что не так, где (файл/строка), как воспроизвести, как предлагаешь исправить. Дубликаты (в том "
    "числе закрытых issues) создавать нельзя, лимит — несколько за прогон.\n"
    + ("- run_in_codespace(command) — проверка в облачной среде GitHub Codespaces с твоими правками (они "
       "переносятся автоматически). Первый вызов занимает 1-3 минуты и тратит квоту владельца репозитория, "
       "поэтому сначала проверяй локально через run_command; Codespaces — только если локально не получается "
       "(нужно окружение из devcontainer, другая версия рантайма, база данных).\n" if (USE_CODESPACES and GITHUB_TOKEN) else "")
    +     f"- run_command выполняется в корне репозитория; лимит одной команды {CMD_MAX_SEC} с, всего {CMD_TOTAL_SEC} с на issue. "
    "Не запускай интерактивные программы, серверы и всё, что не завершается само. Если зависимостей нет — "
    "установи (pip install / npm install), но коротко.\n"
    "- Если после ЧЕСТНОГО изучения issue и кода ты убедился, что правка репозитория НЕ нужна (проблема вне "
    "кода, issue уже решён, просьба неосуществима) — вызови report_no_action_needed(reason) с конкретной "
    "причиной ВМЕСТО report_work. Это проверит независимый ревьюер; отговорка «лень» не пройдёт.\n"
    "- report_work(summary, final=True) вызывай ОДИН раз и только когда сделано ВСЁ, что просит issue. "
    "summary обязан содержать три раздела: «Сделано», «Не сделано / ограничения», «Проверка» (что запускал и "
    "что получилось; если не проверял — так и напиши). Отчёт сверяется с реальным diff: заявлять о том, чего "
    "в diff нет, бесполезно.\n"
    "- Пометка [шаг N/M] в конце результата показывает остаток бюджета раунда. Задача может занять несколько "
    "раундов — это нормально. Не отчитывайся раньше времени: если не закончил, заверши раунд коротким текстом "
    "(что сделано, что осталось)."
)

def task_context(sess):
    return (f"Репозиторий: {GITHUB_OWNER}/{GITHUB_REPO}. Базовая ветка: {sess['base']}. "
            f"Рабочая ветка: {sess['branch']} (уже выбрана).\n\n"
            f"ЗАДАЧА: реши issue #{sess['issue']}. Его текст:\n\n{sess['issue_text']}")

def repo_facts(sess):
    base = sess["base"]
    ahead = _ahead(base)
    changed = git("diff", "--name-only", base).split()
    dirty = bool(git("status", "--porcelain").strip())
    lines = [f"- коммитов в рабочей ветке: {ahead}",
             f"- изменённые файлы относительно {base}: {', '.join(changed[:15]) or 'нет'}",
             f"- есть незакоммиченные правки: {'да' if dirty else 'нет'}",
             f"- финальный отчёт: {'принят' if sess['final'] else 'не принят'}"]
    if sess["commands"]:
        last = sess["commands"][-3:]
        lines.append("- последние команды: " + "; ".join(f"`{c['cmd'][:60]}` → exit {c['exit']}" for c in last))
    return "\n".join(lines)

def format_log(entries, keep=30):
    rows = []
    for i, e in enumerate(entries[-keep:], 1):
        a = json.dumps(e["args"], ensure_ascii=False)[:100] if e["args"] else ""
        rows.append(f"{i}. {e['tool']}({a}) → {e['result'].splitlines()[0][:120] if e['result'] else ''}")
    return "\n".join(rows) or "(вызовов ещё не было)"

def issue_status(sess, base):
    changes = _has_changes(base)
    if sess["final"] and changes:
        return "solved"
    if sess["no_action"] and not changes:
        return "no_action"
    return "partial" if changes else "failed"

def run_round(system, first_msg, tools, budget):
    by_name = {t.__name__: t for t in tools}
    messages = [{"role": "system", "content": system}, {"role": "user", "content": first_msg}]
    last_text, fmt_errors, over = "", 0, 0
    for _ in range(STEPS_PER_ROUND + TEXT_EXTRA_TURNS):
        try:
            out = llm_stream(messages)
        except ContextExceeded:
            return last_text or "(контекст переполнен)", "context"
        text = clean_answer(out["content"])
        used = (out["usage"] or {}).get("prompt_tokens", 0)
        if used > 0.85 * N_CTX:
            return strip_calls(text) or last_text, "context"
        if not text:
            fmt_errors += 1
            print(f"[{_ts()}] пустой ответ модели ({fmt_errors}/3): вероятно, лимит токенов ушёл на размышления")
            if fmt_errors >= 3:
                return last_text, "format"
            messages.append({"role": "user", "content": "Ответ пуст. Отвечай короче и сразу вызови инструмент (блок @@CALL ... @@END)."})
            continue
        last_text = text
        messages.append({"role": "assistant", "content": text})
        name, raw, problem, n_calls = parse_text_call(text)
        if name is None:
            return text, "text"
        t = by_name.get(name)
        kwargs = None
        if problem is None and t is None:
            problem = f"неизвестный инструмент «{name}». Доступны: {', '.join(by_name)}"
        if problem is None:
            try:
                kwargs = coerce_args(t, name, raw)
            except ValueError as e:
                problem = f"{e}. Сигнатура: {tool_signature(t)}"
        if problem is not None:
            fmt_errors += 1
            print(f"[{_ts()}] ошибка формата вызова ({fmt_errors}/3): {problem}")
            if fmt_errors >= 3:
                return strip_calls(text), "format"
            messages.append({"role": "user", "content": f"ОШИБКА ФОРМАТА (инструмент НЕ вызван, шаг не потрачен): {problem}\nПовтори вызов в правильном формате."})
            continue
        fmt_errors = 0
        if budget.remaining == 0:
            over += 1
            if over >= 2:
                return strip_calls(text), "budget"
        result = t(**kwargs)
        note = "\n(В сообщении было несколько блоков @@CALL: выполнен только первый.)" if n_calls > 1 else ""
        messages.append({"role": "user", "content": f"РЕЗУЛЬТАТ {name}:\n{result}{note}\n\nСледующий вызов (блок @@CALL ... @@END) или, если раунд закончен, короткий текстовый итог без блока."})
    return strip_calls(last_text), "turns"

# ---------------- самопроверка (ориентир) ----------------
def judge_result(sess, patch, status):
    if not JUDGE or status not in ("solved", "no_action"):
        return None, ""
    if status == "solved":
        user = (f"ISSUE:\n{sess['issue_text'][:6000]}\n\nОТЧЁТ АГЕНТА:\n{(sess['final'] or '')[:3000]}\n\nDIFF:\n{patch[:20000]}\n\n"
                "Решает ли diff проблему из issue? Ответь двумя строками:\nВЕРДИКТ: ДА / НЕТ / ЧАСТИЧНО\nПОЯСНЕНИЕ: <1-2 предложения>")
    else:
        user = (f"ISSUE:\n{sess['issue_text'][:6000]}\n\nАГЕНТ РЕШИЛ НЕ МЕНЯТЬ КОД. ПРИЧИНА:\n{sess['no_action'][:3000]}\n\n"
                "Обоснована ли причина? Ответь двумя строками:\nВЕРДИКТ: ДА / НЕТ / ЧАСТИЧНО\nПОЯСНЕНИЕ: <1-2 предложения>")
    try:
        out = llm_stream([{"role": "system", "content": "Ты — строгий ревьюер кода. Оценивай по фактам из diff и текста issue, не по уверенности автора."},
                          {"role": "user", "content": user}], max_tokens=500, think=False, show=False)
        txt = clean_answer(out["content"])
    except Exception as e:
        return None, f"ревьюер недоступен: {e}"
    m = re.search(r"ВЕРДИКТ:\s*(ДА|НЕТ|ЧАСТИЧНО)", txt, re.I)
    note = re.sub(r"(?is)^.*?ПОЯСНЕНИЕ:\s*", "", txt).strip()[:600] if "ПОЯСНЕНИЕ" in txt.upper() else txt[:300]
    return (m.group(1).upper() if m else None), note

# ---------------- комментарий-отчёт в issue ----------------
def compose_comment(res, patch):
    head = {"solved": "✅ **Решено локально.** Изменения не отправлены в репозиторий: патч приложен ниже.",
            "partial": "⚠️ **Частично.** Правка есть, но агент не подтвердил завершение работы."}[res["status"]]
    L = [AGENT_MARKER, "### 🤖 Отчёт локальной модели (Qwen3.8-27B)", "", head, ""]
    if res.get("final_report"):
        L += ["**Отчёт агента:**", "", res["final_report"], ""]
    else:
        L += ["**Последние заметки агента (не проверенный факт):**", "", res.get("last_note") or "(нет)", ""]
    L += ["**Изменённые файлы:**", "```", res["diffstat"], "```", ""]
    if res.get("commands"):
        L += ["**Проверки:** " + "; ".join(f"`{c['cmd'][:60]}` → {c['exit']}" for c in res["commands"]), ""]
    if res.get("judge"):
        L += [f"**Самопроверка (та же модель, ориентир):** {res['judge']}. {res.get('judge_note', '')}", ""]
    if res.get("created_issues"):
        L += ["**Заведены новые issues:** " + ", ".join(f"#{c['number']}" if c.get("number") else f"«{c['title']}» (черновик)"
                                                      for c in res["created_issues"]), ""]
    cut = patch[:PATCH_IN_COMMENT_LIMIT]
    cut_note = f"\n[патч обрезан: показано {len(cut)} из {len(patch)} символов; полный патч — у владельца запуска]" if len(patch) > len(cut) else ""
    L += ["<details><summary>Патч (применить: <code>git apply</code>)</summary>", "", "````diff", cut + cut_note, "````", "", "</details>"]
    return "\n".join(L)

def publish_comment(res, patch):
    if res["status"] not in COMMENT_STATUSES or not patch.strip():
        return {"state": "skipped"}
    body = compose_comment(res, patch)
    _draft(f"report-issue-{res['number']}.md", body)
    if not GITHUB_WRITE:
        return {"state": "draft"}
    url, err = gh_upsert_report(res["number"], body)
    return {"state": "posted", "url": url} if url else {"state": "error", "error": err}

# ---------------- один issue ----------------
STOP_TEXT = {"context": "переполнился контекст модели", "format": "модель не смогла соблюсти формат вызовов",
             "budget": "исчерпан бюджет шагов раунда", "turns": "исчерпан лимит ходов раунда", "text": "модель завершила раунд текстом"}

def solve_issue(it, base):
    n, t0 = it["number"], time.time()
    branch = prepare_branch(n, base)
    sess = new_sess(n, base, branch, issue_text(it))
    call_log, trace, carry, why, rounds = [], [], "", "", 0
    for round_num in range(1, MAX_ROUNDS + 1):
        rounds = round_num
        budget = StepBudget(STEPS_PER_ROUND, call_log)
        tools = make_tools(sess, budget)
        system = SYSTEM_PROMPT + "\n\n" + text_protocol_prompt() + build_tool_spec(tools)
        if round_num == 1:
            msg = task_context(sess) + f"\n\nУ тебя до {STEPS_PER_ROUND} вызовов инструментов на раунд и до {MAX_ROUNDS} раундов."
        else:
            msg = (task_context(sess) + f"\n\nРаунд {round_num}/{MAX_ROUNDS}. Новый бюджет: {STEPS_PER_ROUND} вызовов.\n"
                   f"Последние вызовы:\n{format_log(call_log)}\n"
                   + (f"\nТвои заметки в конце прошлого раунда (твои слова, не проверенный факт):\n{carry}\n" if carry else "")
                   + f"\nПРОВЕРЕНО СИСТЕМОЙ (приоритетнее твоих заметок):\n{repo_facts(sess)}\n"
                   "\nНе повторяй уже сделанное. Не считай задачу выполненной, пока финальный отчёт не принят.")
        print(f"\n===== issue #{n} · раунд {round_num}/{MAX_ROUNDS} =====", flush=True)
        text, why = run_round(system, msg, tools, budget)
        carry = (text or "").strip()[-2000:]
        trace.append({"round": round_num, "end_reason": why, "end_text": carry})
        status = issue_status(sess, base)
        if status in ("solved", "no_action"):
            break
    git("add", "-A", "-N")
    patch = git("diff", "--binary", base)
    status = issue_status(sess, base)
    if patch.strip():
        pathlib.Path(f"{OUT_DIR}/patches/issue-{n}.patch").write_text(patch, encoding="utf-8")
    verdict, note = judge_result(sess, patch, status)
    with open(f"{OUT_DIR}/traces/issue-{n}.jsonl", "w", encoding="utf-8") as f:
        for e in call_log:
            f.write(json.dumps({"call": e}, ensure_ascii=False) + "\n")
        for e in trace:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")
    res = {"number": n, "title": it["title"], "url": it["url"], "labels": it["labels"], "status": status,
            "rounds": rounds, "branch": branch, "commits": _ahead(base),
            "files": git("diff", "--name-only", base).split(), "diffstat": git("diff", "--stat", base).strip(),
            "final_report": sess["final"], "no_action_reason": sess["no_action"], "last_note": carry,
            "stop_reason": (STOP_TEXT.get(why, why) if status in ("solved", "no_action") else
                            f"после {rounds} раунд(ов) финальный отчёт не принят; последний раунд: {STOP_TEXT.get(why, why)}"), "judge": verdict, "judge_note": note,
            "commands": sess["commands"][-6:], "patch_file": f"patches/issue-{n}.patch" if patch.strip() else None,
            "elapsed_sec": round(time.time() - t0), "error": None,
            "created_issues": sess["created"], "cs_runs": sess["cs_total"]}
    res["comment"] = publish_comment(res, patch)
    return res

# ---------------- результаты и отчёт ----------------
RESULTS_PATH = f"{OUT_DIR}/results.json"
STATUS_LABEL = {"solved": "✅ решено", "no_action": "➖ правка не нужна", "partial": "⚠️ частично",
                "failed": "❌ не решено", "error": "🛑 ошибка", "skipped": "⏭ не запускалось"}

def load_results():
    return json.load(open(RESULTS_PATH, encoding="utf-8")) if os.path.exists(RESULTS_PATH) else {}

def save_results(results):
    json.dump(results, open(RESULTS_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

def build_report(issues, results):
    total = {k: 0 for k in STATUS_LABEL}
    for it in issues:
        total[results.get(str(it["number"]), {}).get("status", "skipped")] += 1
    L = [f"# Отчёт агента: {GITHUB_OWNER}/{GITHUB_REPO}", "",
         f"Дата: {time.strftime('%Y-%m-%d %H:%M')} · issues в очереди: {len(issues)} · "
         + " · ".join(f"{STATUS_LABEL[k]}: {v}" for k, v in total.items() if v), "",
         "Агент ничего не отправлял в GitHub: правки лежат в патчах (`patches/issue-N.patch`). "
         "Применить: `git checkout -b agent/issue-N && git apply patches/issue-N.patch`.",
         "Статус «решено» значит: есть правка и принят финальный отчёт агента. "
         "«Самопроверка» — мнение той же модели, это ориентир, а не гарантия.", "",
         "| Issue | Статус | Раундов | Коммитов | Файлов | Самопроверка |", "|---|---|---|---|---|---|"]
    for it in issues:
        r = results.get(str(it["number"]))
        if r:
            L.append(f"| #{it['number']} {it['title'][:60]} | {STATUS_LABEL[r['status']]} | {r.get('rounds', '-')} | "
                     f"{r.get('commits', '-')} | {len(r.get('files', []))} | {r.get('judge') or '-'} |")
        else:
            L.append(f"| #{it['number']} {it['title'][:60]} | {STATUS_LABEL['skipped']} | - | - | - | - |")
    if not GH_WRITE["ok"]:
        L += ["", f"> ⚠️ **Запись в GitHub отключилась:** {GH_WRITE['reason']}"]
    if CREATED_ISSUES:
        L += ["", "## Issues, заведённые агентом", ""]
        for c in CREATED_ISSUES:
            L.append(f"- {('#' + str(c['number']) + ' ' + c['url']) if c.get('number') else '(черновик)'} · priority:{c['priority']} · "
                     f"«{c['title']}» · найдено при работе над #{c['found_in']}")
    for it in issues:
        r = results.get(str(it["number"]))
        L += ["", "---", "", f"## #{it['number']} {it['title']}", f"{it['url']}", ""]
        if not r:
            L.append("Не запускалось (закончился лимит времени или issue пропущен).")
            continue
        L.append(f"**Статус:** {STATUS_LABEL[r['status']]} · раундов: {r['rounds']} · время: {r['elapsed_sec']} с")
        if r.get("error"):
            L += ["", f"**Ошибка:** {r['error']}"]
        if r["status"] == "solved":
            L += ["", "**Что сделано (отчёт агента):**", "", r["final_report"] or ""]
        elif r["status"] == "no_action":
            L += ["", "**Агент решил, что правка не нужна. Причина:**", "", r["no_action_reason"] or ""]
        else:
            L += ["", f"**Почему не завершено:** {r.get('stop_reason', '-')}.",
                  "", "**Последние заметки агента (его слова, не проверенный факт):**", "", r.get("last_note") or "(нет)"]
        if r.get("files"):
            L += ["", "**Изменённые файлы:**", "", "```", r["diffstat"], "```"]
            if r.get("patch_file"):
                L.append(f"Патч: `{r['patch_file']}`")
        if r.get("commands"):
            L += ["", "**Последние команды:** " + "; ".join(f"`{c['cmd'][:60]}` → {c['exit']}" for c in r["commands"])]
        if r.get("judge") or r.get("judge_note"):
            L += ["", f"**Самопроверка:** {r.get('judge') or '—'}. {r.get('judge_note', '')}"]
        cm = r.get("comment") or {}
        if cm.get("state") == "posted":
            L += ["", f"**Комментарий в issue:** опубликован ({cm.get('url')})"]
        elif cm.get("state") == "draft":
            L += ["", f"**Комментарий в issue:** черновик в `drafts/report-issue-{r['number']}.md` (GITHUB_WRITE=False)"]
        elif cm.get("state") == "error":
            L += ["", f"**Комментарий в issue:** НЕ опубликован: {cm.get('error')}. Черновик: `drafts/report-issue-{r['number']}.md`"]
        if r.get("created_issues"):
            L += ["", "**Новые issues, заведённые агентом:** " + ", ".join(
                f"#{c['number']} (priority:{c['priority']})" if c.get("number") else f"«{c['title']}» (черновик)" for c in r["created_issues"])]
        if r.get("cs_runs"):
            L += ["", f"**Запусков в Codespaces:** {r['cs_runs']}"]
    return "\n".join(L)

def write_report(issues, results):
    rep = build_report(issues, results)
    pathlib.Path(f"{OUT_DIR}/agent_report.md").write_text(rep, encoding="utf-8")
    return rep

def run_agent():
    t_start = time.time()
    base = clone_repo()
    issues = fetch_issues(ISSUE_STATE)
    if ONLY_ISSUES:
        issues = [i for i in issues if i["number"] in ONLY_ISSUES]
    if ISSUE_LIMIT:
        issues = issues[:ISSUE_LIMIT]
    print(f"База: {base}. Issues в очереди ({len(issues)}): " + ", ".join(f"#{i['number']}" for i in issues))
    results = load_results()
    RUN_CREATED["n"] = 0
    try:
        load_known_issues()
    except Exception as e:
        print(f"Не удалось загрузить список всех issues для защиты от дубликатов: {e}")
    try:
        _run_queue(issues, results, base, t_start)
    finally:
        stop_codespace_if_any()
    rep = write_report(issues, results)
    print("\n" + "=" * 70 + "\n" + rep)
    return results

def _run_queue(issues, results, base, t_start):
    for it in issues:
        key = str(it["number"])
        if key in results and not RERUN_DONE and results[key]["status"] != "error":
            print(f"#{key}: уже обработан ({results[key]['status']}), пропускаю (RERUN_DONE=False)")
            continue
        if MAX_TOTAL_SEC and time.time() - t_start > MAX_TOTAL_SEC:
            print("Достигнут MAX_TOTAL_SEC: новые issues не запускаются.")
            break
        print(f"\n######## issue #{key}: {it['title']} ########", flush=True)
        try:
            res = solve_issue(it, base)
        except Exception as e:
            res = {"number": it["number"], "title": it["title"], "url": it["url"], "labels": it["labels"],
                   "status": "error", "rounds": 0, "commits": 0, "files": [], "elapsed_sec": 0,
                   "error": f"{type(e).__name__}: {str(e)[:300]}"}
        results[key] = res
        save_results(results)
        write_report(issues, results)
        print(f"#{key}: {STATUS_LABEL[res['status']]}")

### Проверка инструментов без модели и без GitHub (рекомендуется, занимает секунды)

In [ ]:
# Проверка инструментов и протокола БЕЗ модели и без GitHub (на временном git-репозитории)
_old_repo_dir = REPO_DIR
REPO_DIR = "/tmp/agent_selftest_repo"
try:
    shutil.rmtree(REPO_DIR, ignore_errors=True); os.makedirs(REPO_DIR)
    subprocess.run("git init -q -b main . 2>/dev/null || (git init -q . && git checkout -q -b main)", shell=True, cwd=REPO_DIR, check=True)
    pathlib.Path(f"{REPO_DIR}/calc.py").write_text("def add(a, b):\n    return a - b\n")
    pathlib.Path(f"{REPO_DIR}/test_calc.py").write_text("from calc import add\nassert add(2, 3) == 5, 'add сломан'\nprint('tests ok')\n")
    git("add", "-A"); git("commit", "-q", "-m", "init")
    sess = new_sess(1, "main", "agent/issue-1", "add() вычитает")
    git("checkout", "-q", "-b", "agent/issue-1")
    T = {t.__name__: t for t in make_tools(sess, StepBudget(60))}
    assert "calc.py" in T["list_files"]()
    assert "return a - b" in T["read_file"]("calc.py")
    assert "calc.py:2" in T["search_code"]("a - b")
    assert "вне репозитория" in T["read_file"]("../etc/passwd") or "ERROR" in T["read_file"]("../etc/passwd")
    assert "заблокирована" in T["run_command"]("git push origin main")
    assert "exit=1" in T["run_command"]("python test_calc.py")
    assert "не найден" in T["patch_file"]("calc.py", "a * b", "a + b")
    assert T["patch_file"]("calc.py", "return a - b", "return a + b").startswith("OK")
    assert "tests ok" in T["run_command"]("python test_calc.py")
    assert "ВНИМАНИЕ" in T["write_file"]("broken.py", "def f(:\n")
    T["delete_file"]("broken.py")
    assert "return a + b" in T["git_diff"]()
    assert T["report_no_action_needed"]("не нужно").startswith("ERROR")      # есть изменения: отказ
    assert T["report_work"]("промежуточно").startswith("OK")
    assert T["report_work"]("Сделано: ...", True).startswith("OK")
    assert _ahead("main") == 1 and issue_status(sess, "main") == "solved"
    name, args, err, n = parse_text_call("@@CALL read_file\n@@ARG path\nx.py\n@@END")
    assert (name, args, err) == ("read_file", {"path": "x.py"}, None)
    assert parse_text_call("@@CALL write_file\n@@ARG path\na")[2] is not None     # блок без @@END не выполняется
    # запись в GitHub: в самотесте работаем только черновиками (без сети)
    _w, _k = GITHUB_WRITE, dict(KNOWN_ISSUES)
    GITHUB_WRITE = False; KNOWN_ISSUES.clear(); KNOWN_ISSUES[7] = "Падает логин при пустом пароле"
    assert "похожий issue уже есть: #7" in T["create_issue"]("Логин падает при пустом пароле", "x" * 60)
    assert "слишком" in T["create_issue"]("коротко", "x") or "ERROR" in T["create_issue"]("коротко", "x")
    assert T["create_issue"]("Нет проверки деления на ноль", "В calc.py нет функции div; деление на ноль не обработано. Воспроизвести: div(1, 0).", "p1").startswith("OK")
    assert "похожий issue уже есть" in T["create_issue"]("Нет проверки деления на ноль", "y" * 60)       # второй раз - дубликат
    assert T["comment_issue"]("Уточните, нужна ли поддержка float? Сейчас add принимает любые типы.").startswith("OK")
    assert "ERROR" in T["create_issue"]("Ошибка в стиле кода", "z" * 60, "p9")
    CREATED_ISSUES.clear(); GITHUB_WRITE = _w; KNOWN_ISSUES.clear(); KNOWN_ISSUES.update(_k)
    assert "base64 -d" in _codespace_script("pytest -q", "QUJD", 60) and "timeout -k 5 60" in _codespace_script("pytest -q", "QUJD", 60)
    print("✅ Инструменты и протокол работают.")
finally:
    REPO_DIR = _old_repo_dir

## 4. Запуск

Сначала посмотрите очередь issues (проверяет доступ к GitHub), затем запускайте агента. Для первого пробного прогона удобно
поставить `ISSUE_LIMIT = 1` или `ONLY_ISSUES = [номер]` в настройках агента и перезапустить ячейку настроек.
Если сессия оборвётся, запустите ячейку запуска снова: уже обработанные issues пропускаются (`results.json`).

In [ ]:
queue = fetch_issues(ISSUE_STATE)
print(f"Issues ({ISSUE_STATE}): {len(queue)}")
for it in queue[:50]:
    print(f"#{it['number']:<5} {it['title'][:80]}  [{','.join(it['labels']) or '-'}]")

In [ ]:
results = run_agent()

## 5. Остановить сервер (освободить GPU)

In [ ]:
stop_server()